# 05 — Gold Aggregations

**Week:** 7  
**Date:** September 1–7, 2026

**Goal:** Create dashboard-ready Gold tables and document KPI formulas and metric grain.


## Trusted data used

- **`trusted_shipments`**

The Gold tables below use DQ-validated shipment records from the trusted layer.


In [ ]:
trusted_shipments = spark.table("trusted_shipments")
silver_hubs = spark.table("silver_hubs")


## GOLD-01 — Daily shipment metrics

**`gold_shipment_daily_metrics`** provides daily shipment KPIs at the booking date grain.


In [ ]:
%sql
CREATE OR REPLACE TABLE gold_shipment_daily_metrics AS
SELECT
    CAST(booking_ts AS DATE) AS metric_date,
    COUNT(*) AS total_shipments,
    SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts <= promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS on_time_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts > promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS delayed_deliveries,
    ROUND(
        100.0 * SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS delivery_rate_pct,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN UPPER(delivery_outcome) = 'DELIVERED'
                 AND actual_delivery_ts <= promised_delivery_ts
                THEN 1
                ELSE 0
            END
        )
        / NULLIF(SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END), 0),
        2
    ) AS on_time_delivery_rate_pct,
    ROUND(
        AVG(
            CASE
                WHEN actual_delivery_ts IS NOT NULL
                THEN (unix_timestamp(actual_delivery_ts) - unix_timestamp(pickup_ts)) / 3600.0
            END
        ),
        2
    ) AS avg_delivery_hours,
    ROUND(SUM(freight_amount_inr), 2) AS total_freight_amount_inr
FROM trusted_shipments
GROUP BY CAST(booking_ts AS DATE);


## GOLD-02 — Carrier metrics

**`gold_carrier_metrics`** provides carrier-level shipment and delivery performance.


In [ ]:
%sql
CREATE OR REPLACE TABLE gold_carrier_metrics AS
SELECT
    carrier_id,
    COUNT(*) AS total_shipments,
    SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts <= promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS on_time_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts > promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS delayed_deliveries,
    ROUND(
        100.0 * SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS delivery_rate_pct,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN UPPER(delivery_outcome) = 'DELIVERED'
                 AND actual_delivery_ts <= promised_delivery_ts
                THEN 1
                ELSE 0
            END
        )
        / NULLIF(SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END), 0),
        2
    ) AS on_time_delivery_rate_pct,
    ROUND(
        AVG(
            CASE
                WHEN actual_delivery_ts IS NOT NULL
                THEN (unix_timestamp(actual_delivery_ts) - unix_timestamp(pickup_ts)) / 3600.0
            END
        ),
        2
    ) AS avg_delivery_hours,
    ROUND(SUM(freight_amount_inr), 2) AS total_freight_amount_inr
FROM trusted_shipments
GROUP BY carrier_id;


## GOLD-03 — Route metrics

**`gold_route_metrics`** provides route-level shipment and delivery performance.


In [ ]:
%sql
CREATE OR REPLACE TABLE gold_route_metrics AS
SELECT
    route_id,
    COUNT(*) AS total_shipments,
    SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts <= promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS on_time_shipments,
    SUM(
        CASE
            WHEN UPPER(delivery_outcome) = 'DELIVERED'
             AND actual_delivery_ts > promised_delivery_ts
            THEN 1
            ELSE 0
        END
    ) AS delayed_deliveries,
    ROUND(
        100.0 * SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS delivery_rate_pct,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN UPPER(delivery_outcome) = 'DELIVERED'
                 AND actual_delivery_ts <= promised_delivery_ts
                THEN 1
                ELSE 0
            END
        )
        / NULLIF(SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END), 0),
        2
    ) AS on_time_delivery_rate_pct,
    ROUND(AVG(package_weight_kg), 2) AS avg_package_weight_kg,
    ROUND(SUM(freight_amount_inr), 2) AS total_freight_amount_inr
FROM trusted_shipments
GROUP BY route_id;


## GOLD-04 — Hub metrics

**`gold_hub_metrics`** combines origin and destination shipment activity at the hub grain.


In [ ]:
%sql
CREATE OR REPLACE TABLE gold_hub_metrics AS
WITH origin_metrics AS (
    SELECT
        origin_hub_id AS hub_id,
        COUNT(*) AS origin_shipments,
        0 AS destination_shipments,
        SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END) AS origin_delivered
    FROM trusted_shipments
    GROUP BY origin_hub_id
),
destination_metrics AS (
    SELECT
        destination_hub_id AS hub_id,
        0 AS origin_shipments,
        COUNT(*) AS destination_shipments,
        SUM(CASE WHEN UPPER(delivery_outcome) = 'DELIVERED' THEN 1 ELSE 0 END) AS destination_delivered
    FROM trusted_shipments
    GROUP BY destination_hub_id
),
combined AS (
    SELECT * FROM origin_metrics
    UNION ALL
    SELECT * FROM destination_metrics
)
SELECT
    hub_id,
    SUM(origin_shipments) AS origin_shipments,
    SUM(destination_shipments) AS destination_shipments,
    SUM(origin_delivered) AS delivered_shipments,
    SUM(origin_shipments) + SUM(destination_shipments) AS total_hub_shipments
FROM combined
GROUP BY hub_id;


## Gold table validation

The following checks verify that the Gold tables were created successfully and do not contain unexpected nulls or invalid metric values.


In [ ]:
%sql
SELECT
    'gold_shipment_daily_metrics' AS table_name,
    COUNT(*) AS row_count
FROM gold_shipment_daily_metrics

UNION ALL

SELECT
    'gold_carrier_metrics',
    COUNT(*)
FROM gold_carrier_metrics

UNION ALL

SELECT
    'gold_route_metrics',
    COUNT(*)
FROM gold_route_metrics

UNION ALL

SELECT
    'gold_hub_metrics',
    COUNT(*)
FROM gold_hub_metrics;


## Gold output examples

The following queries return the dashboard-ready Gold records for review.


In [ ]:
%sql
SELECT *
FROM gold_shipment_daily_metrics
ORDER BY metric_date;


In [ ]:
%sql
SELECT *
FROM gold_carrier_metrics
ORDER BY total_shipments DESC;


In [ ]:
%sql
SELECT *
FROM gold_route_metrics
ORDER BY total_shipments DESC;


In [ ]:
%sql
SELECT
    h.hub_id,
    h.hub_name,
    h.city,
    h.region,
    g.origin_shipments,
    g.destination_shipments,
    g.delivered_shipments,
    g.total_hub_shipments
FROM gold_hub_metrics g
LEFT JOIN silver_hubs h
    ON g.hub_id = h.hub_id
ORDER BY total_hub_shipments DESC;


## Final Gold results summary

Run the validation cells above after creating all four Gold tables. The output can be used for the Week 7 dashboard evidence and KPI documentation.

The four Gold tables provide dashboard-ready metrics for daily shipment performance, carrier performance, route performance, and hub shipment flow.


In [ ]:
%sql
SELECT
    'gold_shipment_daily_metrics' AS table_name,
    COUNT(*) AS row_count
FROM gold_shipment_daily_metrics
UNION ALL
SELECT 'gold_carrier_metrics', COUNT(*)
FROM gold_carrier_metrics
UNION ALL
SELECT 'gold_route_metrics', COUNT(*)
FROM gold_route_metrics
UNION ALL
SELECT 'gold_hub_metrics', COUNT(*)
FROM gold_hub_metrics;
